## LOAD DATASET

In [ ]:
# 1/ Data import

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
file_path = "../data/metabo_all_act1.xlsx"
df = pd.read_excel(file_path)

# Combine ScFv, Donor, and Costim into a new column representing the condition
df["Condition"] = (
    df["ScFv"]
    + "_"
    + df["Donor"].astype(str)
    + "_"
    + df["Costim"]
)

# Verify the new column
print(df[["ScFv", "Donor", "Costim", "Condition"]])

## ALL DATA PER CONSTRUCT

In [ ]:
from sklearn.preprocessing import StandardScaler

# List of numerical columns
numerical_cols = [
    "OCR", "ECAR", "SRC", "Puro", "Mit_ dep",
    "GLUT1", "ASCT2", "MTG", "TMRM", "MitoSox"
]

# Step 1: Combine ScFv and Costim into a new condition column
df["ScFv_Costim"] = df["ScFv"] + "_" + df["Costim"]

# Step 2: Apply Z-score normalization across all numerical columns
scaler = StandardScaler()

df_normalized = df.copy()

df_normalized[numerical_cols] = scaler.fit_transform(
    df[numerical_cols]
)

# Step 3: Reshape into long-form for Seaborn
df_melted = df_normalized.melt(
    id_vars=["ScFv_Costim"],
    value_vars=numerical_cols,
    var_name="Parameter",
    value_name="Normalized_Value"
)

# Customizable parameters
figure_width = 15
figure_height = 5

# Custom colors for each ScFv_Costim condition
colors = {
    "CD19_CD28": "darkred",
    "CD19_41BB": "red",
    "CD22_CD28": "darkblue",
    "CD22_41BB": "blue",
    "CD33_CD28": "darkgreen",
    "CD33_41BB": "green"
}

# Step 4: Plot
plt.figure(figsize=(figure_width, figure_height))

sns.boxplot(
    x="Parameter",
    y="Normalized_Value",
    hue="ScFv_Costim",
    data=df_melted,
    palette=colors,
    showfliers=False
)

sns.stripplot(
    x="Parameter",
    y="Normalized_Value",
    data=df_melted,
    hue="ScFv_Costim",
    palette=colors,
    dodge=True,
    jitter=True,
    marker="o",
    color="black",
    size=4,
    alpha=0.6,
    legend=False
)

plt.title("", fontsize=16)
plt.xlabel("Numerical Parameters", fontsize=14)
plt.ylabel("Normalized Value (Z-Score)", fontsize=14)

plt.ylim(-2.5, 3)

plt.legend(
    title="ScFv_Costim Condition",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=12
)

plt.tight_layout()
plt.show()

## CORRELATION MATRIX - ALL POINTS


In [ ]:
# Pairwise metabolic parameter relationships

params = [
    "OCR", "ECAR", "SRC", "GLUT1",
    "ASCT2", "MTG", "TMRM", "MitoSox"
]

colors = {
    "CD19_CD28": "darkred",
    "CD19_41BB": "red",
    "CD22_CD28": "darkblue",
    "CD22_41BB": "blue",
    "CD33_CD28": "darkgreen",
    "CD33_41BB": "green"
}

shapes = {
    "A": "o",
    "B": "s",
    "C": "P",
    "D": "X"
}

# Data preparation
data = df.copy()

if "ScFv_Costim" not in data.columns:
    data["ScFv_Costim"] = (
        data["ScFv"].astype(str)
        + "_"
        + data["Costim"].astype(str)
    )

data = data[
    params + ["ScFv_Costim", "Donor"]
].dropna(subset=params)


# Scatter plots with global linear regression
def scatter_groups_with_global_reg(x, y, **kws):
    ax = plt.gca()

    scfv = data.loc[x.index, "ScFv_Costim"]
    donor = data.loc[x.index, "Donor"]

    for scfv_key, color in colors.items():
        for donor_key, marker in shapes.items():

            mask = (
                (scfv == scfv_key)
                & (donor == donor_key)
            )

            if mask.any():
                ax.scatter(
                    x[mask],
                    y[mask],
                    c=color,
                    marker=marker
                )

    valid = x.notna() & y.notna()

    if valid.sum() >= 3:
        xs = x[valid].to_numpy()
        ys = y[valid].to_numpy()

        m, b = np.polyfit(xs, ys, 1)

        xs_sorted = np.sort(xs)
        y_line = m * xs_sorted + b

        y_pred = m * xs + b
        residuals = ys - y_pred
        sd = np.std(residuals)

        ax.fill_between(
            xs_sorted,
            y_line - sd,
            y_line + sd,
            alpha=0.15
        )

        ax.plot(
            xs_sorted,
            y_line
        )


# Diagonal
def diag_scatter_same(x, **kws):
    ax = plt.gca()

    scfv = data.loc[x.index, "ScFv_Costim"]
    donor = data.loc[x.index, "Donor"]

    for scfv_key, color in colors.items():
        for donor_key, marker in shapes.items():

            mask = (
                (scfv == scfv_key)
                & (donor == donor_key)
            )

            if mask.any():
                ax.scatter(
                    x[mask],
                    x[mask],
                    c=color,
                    marker=marker
                )

    valid = x.notna()

    if valid.sum() >= 2:
        xs = x[valid].to_numpy()
        lo, hi = xs.min(), xs.max()

        ax.plot(
            [lo, hi],
            [lo, hi]
        )


# Pairwise plot
g = sns.PairGrid(
    data=data,
    vars=params,
    diag_sharey=False
)

g.map_lower(scatter_groups_with_global_reg)
g.map_upper(scatter_groups_with_global_reg)
g.map_diag(diag_scatter_same)

plt.show()

## METABOLIC SCORE

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Metabolic parameters included in the PCA
metabolic_cols = [
    "OCR", "ECAR", "SRC", "GLUT1",
    "ASCT2", "MTG", "TMRM", "MitoSox"
]

# Standardize metabolic parameters
X = df[metabolic_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Metabolic PCA score = PC1
df["Metabolic_PCA_Score"] = X_pca[:, 0]

In [ ]:
# Metabolic PCA score distribution

colors = {
    "CD19_CD28": "darkred",
    "CD19_41BB": "red",
    "CD22_CD28": "darkblue",
    "CD22_41BB": "dodgerblue",
    "CD33_CD28": "darkgreen",
    "CD33_41BB": "mediumseagreen"
}

shapes = {
    "A": "o",
    "B": "s",
    "C": "P",
    "D": "X"
}

fig, ax = plt.subplots()

# Horizontal violin
sns.violinplot(
    ax=ax,
    y=["all"] * len(df),
    x="Metabolic_PCA_Score",
    data=df,
    inner=None,
    color="whitesmoke"
)

# Overlay points
sns.scatterplot(
    ax=ax,
    y=["all"] * len(df),
    x=df["Metabolic_PCA_Score"],
    hue=df["ScFv_Costim"],
    style=df["Donor"],
    palette=colors,
    markers=shapes,
    legend=False
)

ax.set_title("Metabolic PCA score")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticks([])

plt.tight_layout()
plt.show()